# 使用已训练的 XLNet 模型（简洁版）

首次按顺序运行下面三个代码单元；之后只修改第三个单元中的文字。
本文件不训练、不下载模型、不修改模型文件，支持英文的快乐、愤怒、恐惧、悲伤分类，没有中性类别。


## 1. 导入依赖，指定模型目录

默认使用之前准确率约 82.55% 的全量模型，不是 quick 调试模型。

In [1]:
import os
# 保留当前 Windows 环境所需设置，必须先于模型库导入。
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import json
import runpy
from pathlib import Path
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoTokenizer, XLNetForSequenceClassification

# ⭐ 换模型时，只修改这里：指向自己训练保存的 final_model 目录。
MODEL_DIR = Path(
    r"D:\Study\AI_Study\Course1_365\34_LLMs使用XLNet进行文本分类"
    r"\xlnet_runs\20260917_204451_291433\final_model"
)


## 2. 加载一次模型，定义预测函数

清洗规则和截断长度直接复用训练时保存的文件。只使用自己信任的目录：`preprocessing.py` 会作为 Python 代码执行。

In [2]:
# 读取训练时的清洗函数与参数，避免预测时的处理方式和训练不一致。
clean_text = runpy.run_path(str(MODEL_DIR / "preprocessing.py"))["clean_for_xlnet"]
config = json.loads((MODEL_DIR / "preprocessing_config.json").read_text(encoding="utf-8"))

# 只从本地加载，默认使用 CPU；eval() 关闭训练时的 dropout。
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = XLNetForSequenceClassification.from_pretrained(
    MODEL_DIR, local_files_only=True, use_safetensors=True
)
model.eval()
model.config.use_mems_eval = False  # 不把上一批文本的记忆传给下一批。


def predict_emotions(texts):
    """输入一条英文字符串或字符串列表，返回情绪和分数。适合少量文本。"""
    if isinstance(texts, str):
        texts = [texts]
    if not isinstance(texts, list) or not texts or not all(isinstance(t, str) for t in texts):
        raise ValueError("请输入字符串或非空字符串列表。")

    # ① 清洗文字：与训练时一样去除 @用户名、emoji 等。
    cleaned = [clean_text(text) for text in texts]
    if any(not text.strip() for text in cleaned):
        raise ValueError("清洗后文本为空，请输入有实际文字的内容。")

    # ② 分词：把文字转成数字；补齐批次长度，过长文本按训练时的上限截断。
    inputs = tokenizer(
        cleaned, padding=True, truncation=True,
        max_length=config["max_length"], return_tensors="pt",
    )

    # ③ 预测：不记录梯度、不更新权重；softmax 将分类分数转为概率形式。
    with torch.inference_mode():
        probabilities = model(**inputs).logits.softmax(dim=-1)
    scores, ids = probabilities.max(dim=-1)

    # ④ 将最大分数对应的编号翻译回标签；分数不等于保证正确的概率。
    return pd.DataFrame({
        "text": texts,
        "emotion": [model.config.id2label[i] for i in ids.tolist()],
        "confidence": scores.tolist(),
    })

print("模型加载完成，可以开始预测。")


Since the GPL-licensed package `unidecode` is not installed, using Python's `unicodedata` package which yields worse results.


模型加载完成，可以开始预测。


## 3. ⭐ 修改这里的文字即可

单条：`predict_emotions("I am happy!")`；多条：传入下面这样的列表。
`joy` 快乐 · `anger` 愤怒 · `sadness` 悲伤 · `fear` 恐惧。大量文本应拆成小列表分次预测。

In [3]:
TEXTS = [
    "@friend I am so happy and excited about this wonderful news! ❤️",
    "I am furious. This is completely unfair!",
    "I feel lonely and sad. I miss my family.",
    "I am scared and worried about what will happen tomorrow.",
]
display(predict_emotions(TEXTS))


,text,emotion,confidence
0,@friend I am so happy and excited about this w...,joy,0.991676
1,I am furious. This is completely unfair!,anger,0.994162
2,I feel lonely and sad. I miss my family.,sadness,0.987902
3,I am scared and worried about what will happen...,fear,0.992119
